# Inspecting the evidence behind the two stories

Fide AI, September 26, 2026. This notebook checks the **derived tables** supplied in the companion package. It does not authenticate them against original archive JSON or estimate incident prevalence. It can run from the package root using Python 3.11+ and the standard library. See METHODS.md for source verification and missing-body limitations.


In [1]:
from pathlib import Path
import csv, json
from collections import Counter
ROOT = Path.cwd()
assert (ROOT / 'results/archive-observations.csv').exists(), 'Run from the extracted package root'
def rows(name):
    with (ROOT / 'results' / name).open() as f:
        return list(csv.DictReader(f))
observations = rows('archive-observations.csv')
transactions = rows('archive-transactions.csv')
assert len({r['report_id'] for r in observations}) == len(observations)
assert len({(r['report_id'], r['index']) for r in transactions}) == len(transactions)
assert {r['report_id'] for r in transactions} <= {r['report_id'] for r in observations}
print('Selected reports:', len(observations), '| HTTP entries:', len(transactions))
print('Report cohorts:', dict(Counter(r['cohort'] for r in observations)))

Selected reports: 187 | HTTP entries: 2978
Report cohorts: {'upstream_background_challenge': 79, 'reserved_family_transfer': 79, 'development_known_oncb': 29}


In [2]:
bodies = sum(r['body_in_json'] == 'True' for r in transactions)
anomalies = json.loads((ROOT / 'results/hash-field-anomalies.json').read_text())
print(f'Raw response bodies present: {bodies}/{len(transactions)}')
print(f'Positive-size entries with empty-string hash: {len(anomalies)}/{len(transactions)}')
print('These missing fields prevent exact source-to-output byte comparisons.')

Raw response bodies present: 0/2978
Positive-size entries with empty-string hash: 128/2978
These missing fields prevent exact source-to-output byte comparisons.


In [3]:
decisions = rows('transfer-decisions.csv')
receipts = rows('transfer-receipts.csv')
cases = {c['id']: c for c in json.loads((ROOT / 'data/transfer-cases.json').read_text())}
for condition in ('none', 'browser_only', 'per_tool', 'shared'):
    group = [r for r in decisions if r['condition'] == condition]
    completed = 0
    for r in group:
        final = [j for j in receipts if j['trial'] == r['trial'] and j['recipient'] == cases[r['case']]['destination']]
        full = any(j['sha256'] == r['intended_sha256'] and int(j['bytes']) == int(r['intended_bytes']) for j in final)
        assert full == (r['final_full_receipt'] == 'True')
        completed += full and r['expected_authorized'] == 'True'
    print(condition, ':', completed, 'permitted complete transfers; authored cases, not a field rate')

none : 19 permitted complete transfers; authored cases, not a field rate
browser_only : 18 permitted complete transfers; authored cases, not a field rate
per_tool : 16 permitted complete transfers; authored cases, not a field rate
shared : 13 permitted complete transfers; authored cases, not a field rate
